In [1]:
import pandas as pd
import requests
from pathlib import Path

# ------------------------------------------------------
# 1. Paste your ERDDAP htmlTable URL here
# ------------------------------------------------------
html_url = r"""https://www.ncei.noaa.gov/erddap/griddap/ncdc_oisst_v2_avhrr_by_time_zlev_lat_lon.htmlTable?ice%5B(2026-04-27T12:00:00Z):1:(2026-04-27T12:00:00Z)%5D%5B(0.0):1:(0.0)%5D%5B(-89.875):1:(89.875)%5D%5B(0.125):1:(359.875)%5D,err%5B(2026-04-27T12:00:00Z):1:(2026-04-27T12:00:00Z)%5D%5B(0.0):1:(0.0)%5D%5B(-89.875):1:(89.875)%5D%5B(0.125):1:(359.875)%5D,anom%5B(2026-04-27T12:00:00Z):1:(2026-04-27T12:00:00Z)%5D%5B(0.0):1:(0.0)%5D%5B(-89.875):1:(89.875)%5D%5B(0.125):1:(359.875)%5D,sst%5B(2026-04-27T12:00:00Z):1:(2026-04-27T12:00:00Z)%5D%5B(0.0):1:(0.0)%5D%5B(-89.875):1:(89.875)%5D%5B(0.125):1:(359.875)%5D"""

# ------------------------------------------------------
# 2. Convert htmlTable URL to CSVP URL
# CSVP gives a clean table format suitable for pandas
# ------------------------------------------------------
csv_url = html_url.replace(".htmlTable?", ".csvp?")

print("CSV URL:")
print(csv_url)

# ------------------------------------------------------
# 3. Output files
# ------------------------------------------------------
output_dir = Path("noaa_oisst_output")
output_dir.mkdir(exist_ok=True)

csv_file = output_dir / "noaa_oisst_20260427_global.csv"
excel_file = output_dir / "noaa_oisst_20260427_global.xlsx"

# ------------------------------------------------------
# 4. Download CSV file
# ------------------------------------------------------
response = requests.get(csv_url, stream=True, timeout=300)
response.raise_for_status()

with open(csv_file, "wb") as f:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)

print("CSV downloaded:", csv_file)

# ------------------------------------------------------
# 5. Read CSV into pandas
# ------------------------------------------------------
df = pd.read_csv(csv_file)

print("Shape:", df.shape)
print(df.head())

# ------------------------------------------------------
# 6. Clean column names
# ------------------------------------------------------
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"\(|\)", "", regex=True)
    .str.replace("/", "_", regex=True)
)

print("Cleaned columns:")
print(df.columns.tolist())

# ------------------------------------------------------
# 7. Save as Excel
# ------------------------------------------------------
df.to_excel(excel_file, index=False)

print("Excel saved:", excel_file)

CSV URL:
https://www.ncei.noaa.gov/erddap/griddap/ncdc_oisst_v2_avhrr_by_time_zlev_lat_lon.csvp?ice%5B(2026-04-27T12:00:00Z):1:(2026-04-27T12:00:00Z)%5D%5B(0.0):1:(0.0)%5D%5B(-89.875):1:(89.875)%5D%5B(0.125):1:(359.875)%5D,err%5B(2026-04-27T12:00:00Z):1:(2026-04-27T12:00:00Z)%5D%5B(0.0):1:(0.0)%5D%5B(-89.875):1:(89.875)%5D%5B(0.125):1:(359.875)%5D,anom%5B(2026-04-27T12:00:00Z):1:(2026-04-27T12:00:00Z)%5D%5B(0.0):1:(0.0)%5D%5B(-89.875):1:(89.875)%5D%5B(0.125):1:(359.875)%5D,sst%5B(2026-04-27T12:00:00Z):1:(2026-04-27T12:00:00Z)%5D%5B(0.0):1:(0.0)%5D%5B(-89.875):1:(89.875)%5D%5B(0.125):1:(359.875)%5D
CSV downloaded: noaa_oisst_output/noaa_oisst_20260427_global.csv
Shape: (1036800, 8)
             time (UTC)  depth (m)  latitude (degrees_north)  \
0  2026-04-27T12:00:00Z        0.0                   -89.875   
1  2026-04-27T12:00:00Z        0.0                   -89.875   
2  2026-04-27T12:00:00Z        0.0                   -89.875   
3  2026-04-27T12:00:00Z        0.0                   -

In [2]:
!pip install pandas openpyxl requests tqdm -q

In [3]:
import pandas as pd
import requests
from pathlib import Path
from tqdm import tqdm
from calendar import monthrange

# ============================================================
# NOAA OISST ERDDAP Dataset
# ============================================================

BASE_URL = "https://www.ncei.noaa.gov/erddap/griddap/ncdc_oisst_v2_avhrr_by_time_zlev_lat_lon.csvp"

OUTPUT_DIR = Path("noaa_oisst_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Study regions
# ============================================================

REGIONS = {
    "bob": {
        "name": "Bay_of_Bengal",
        "lat_min": 5.125,
        "lat_max": 21.875,
        "lon_min": 80.125,
        "lon_max": 97.875
    },
    "as": {
        "name": "Arabian_Sea",
        "lat_min": 5.125,
        "lat_max": 21.875,
        "lon_min": 60.125,
        "lon_max": 74.875
    }
}

VARIABLES = ["sst", "anom", "err"]

START_YEAR = 2020
END_YEAR = 2024


# ============================================================
# Build ERDDAP URL
# ============================================================

def build_oisst_url(region, start_date, end_date):
    parts = []

    for var in VARIABLES:
        part = (
            f"{var}"
            f"[({start_date}T12:00:00Z):1:({end_date}T12:00:00Z)]"
            f"[(0.0):1:(0.0)]"
            f"[({region['lat_min']}):1:({region['lat_max']})]"
            f"[({region['lon_min']}):1:({region['lon_max']})]"
        )
        parts.append(part)

    return BASE_URL + "?" + ",".join(parts)


# ============================================================
# Download one monthly subset
# ============================================================

def download_monthly_oisst(region_key, year, month):
    region = REGIONS[region_key]

    start_date = f"{year}-{month:02d}-01"
    last_day = monthrange(year, month)[1]
    end_date = f"{year}-{month:02d}-{last_day:02d}"

    url = build_oisst_url(region, start_date, end_date)

    region_dir = OUTPUT_DIR / region["name"]
    csv_dir = region_dir / "csv"
    excel_dir = region_dir / "excel"

    csv_dir.mkdir(parents=True, exist_ok=True)
    excel_dir.mkdir(parents=True, exist_ok=True)

    csv_file = csv_dir / f"oisst_{region_key}_{year}_{month:02d}.csv"
    excel_file = excel_dir / f"oisst_{region_key}_{year}_{month:02d}.xlsx"

    print(f"\nDownloading: {region['name']} | {year}-{month:02d}")
    print(url)

    response = requests.get(url, timeout=600)
    response.raise_for_status()

    with open(csv_file, "wb") as f:
        f.write(response.content)

    df = pd.read_csv(csv_file)

    # Clean column names
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(r"\(|\)", "", regex=True)
        .str.replace("/", "_", regex=True)
    )

    # Add basin column
    df["basin"] = region["name"]

    # Save cleaned CSV again
    df.to_csv(csv_file, index=False)

    # Save Excel
    df.to_excel(excel_file, index=False)

    print("Rows:", len(df))
    print("Saved CSV:", csv_file)
    print("Saved Excel:", excel_file)

    return df


# ============================================================
# Download all monthly OISST files
# ============================================================

for region_key in REGIONS.keys():
    for year in range(START_YEAR, END_YEAR + 1):
        for month in range(1, 13):
            try:
                download_monthly_oisst(region_key, year, month)
            except Exception as e:
                print(f"Failed: {region_key}, {year}-{month:02d}")
                print("Reason:", e)


Downloading: Bay_of_Bengal | 2020-01
https://www.ncei.noaa.gov/erddap/griddap/ncdc_oisst_v2_avhrr_by_time_zlev_lat_lon.csvp?sst[(2020-01-01T12:00:00Z):1:(2020-01-31T12:00:00Z)][(0.0):1:(0.0)][(5.125):1:(21.875)][(80.125):1:(97.875)],anom[(2020-01-01T12:00:00Z):1:(2020-01-31T12:00:00Z)][(0.0):1:(0.0)][(5.125):1:(21.875)][(80.125):1:(97.875)],err[(2020-01-01T12:00:00Z):1:(2020-01-31T12:00:00Z)][(0.0):1:(0.0)][(5.125):1:(21.875)][(80.125):1:(97.875)]
Failed: bob, 2020-01
Reason: 404 Client Error:  for url: https://www.ncei.noaa.gov/erddap/griddap/ncdc_oisst_v2_avhrr_by_time_zlev_lat_lon.csvp?sst%5B(2020-01-01T12:00:00Z):1:(2020-01-31T12:00:00Z)%5D%5B(0.0):1:(0.0)%5D%5B(5.125):1:(21.875)%5D%5B(80.125):1:(97.875)%5D,anom%5B(2020-01-01T12:00:00Z):1:(2020-01-31T12:00:00Z)%5D%5B(0.0):1:(0.0)%5D%5B(5.125):1:(21.875)%5D%5B(80.125):1:(97.875)%5D,err%5B(2020-01-01T12:00:00Z):1:(2020-01-31T12:00:00Z)%5D%5B(0.0):1:(0.0)%5D%5B(5.125):1:(21.875)%5D%5B(80.125):1:(97.875)%5D

Downloading: Bay_of_Bengal

In [4]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("noaa_oisst_output")

all_csv_files = sorted(OUTPUT_DIR.glob("*/*/oisst_*.csv"))

print("Total CSV files found:", len(all_csv_files))

frames = []

for file in all_csv_files:
    print("Reading:", file)
    df = pd.read_csv(file)
    frames.append(df)

df_all = pd.concat(frames, ignore_index=True)

print("Final shape:", df_all.shape)
print(df_all.head())

# Save final research-ready files
final_csv = OUTPUT_DIR / "oisst_bob_as_2020_2024_combined.csv"
final_parquet = OUTPUT_DIR / "oisst_bob_as_2020_2024_combined.parquet"

df_all.to_csv(final_csv, index=False)
df_all.to_parquet(final_parquet, index=False)

print("Saved final CSV:", final_csv)
print("Saved final Parquet:", final_parquet)

Total CSV files found: 116
Reading: noaa_oisst_output/Arabian_Sea/csv/oisst_as_2020_03.csv
Reading: noaa_oisst_output/Arabian_Sea/csv/oisst_as_2020_04.csv
Reading: noaa_oisst_output/Arabian_Sea/csv/oisst_as_2020_05.csv
Reading: noaa_oisst_output/Arabian_Sea/csv/oisst_as_2020_06.csv
Reading: noaa_oisst_output/Arabian_Sea/csv/oisst_as_2020_07.csv
Reading: noaa_oisst_output/Arabian_Sea/csv/oisst_as_2020_08.csv
Reading: noaa_oisst_output/Arabian_Sea/csv/oisst_as_2020_09.csv
Reading: noaa_oisst_output/Arabian_Sea/csv/oisst_as_2020_10.csv
Reading: noaa_oisst_output/Arabian_Sea/csv/oisst_as_2020_11.csv
Reading: noaa_oisst_output/Arabian_Sea/csv/oisst_as_2020_12.csv
Reading: noaa_oisst_output/Arabian_Sea/csv/oisst_as_2021_01.csv
Reading: noaa_oisst_output/Arabian_Sea/csv/oisst_as_2021_02.csv
Reading: noaa_oisst_output/Arabian_Sea/csv/oisst_as_2021_03.csv
Reading: noaa_oisst_output/Arabian_Sea/csv/oisst_as_2021_04.csv
Reading: noaa_oisst_output/Arabian_Sea/csv/oisst_as_2021_05.csv
Reading: noaa

In [5]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("noaa_oisst_output")

df_all = pd.read_parquet(OUTPUT_DIR / "oisst_bob_as_2020_2024_combined.parquet")

summary = (
    df_all.groupby(["basin", "time_utc"], as_index=False)
    .agg(
        sst_mean=("sst_celsius", "mean"),
        sst_min=("sst_celsius", "min"),
        sst_max=("sst_celsius", "max"),
        sst_std=("sst_celsius", "std"),
        anom_mean=("anom_celsius", "mean"),
        err_mean=("err_celsius", "mean"),
        valid_grid_count=("sst_celsius", "count")
    )
)

summary_csv = OUTPUT_DIR / "oisst_daily_basin_summary_2020_2024.csv"
summary_excel = OUTPUT_DIR / "oisst_daily_basin_summary_2020_2024.xlsx"
summary_parquet = OUTPUT_DIR / "oisst_daily_basin_summary_2020_2024.parquet"

summary.to_csv(summary_csv, index=False)
summary.to_excel(summary_excel, index=False)
summary.to_parquet(summary_parquet, index=False)

print("Summary shape:", summary.shape)
print("Saved:", summary_excel)

summary.head()

Summary shape: (3534, 9)
Saved: noaa_oisst_output/oisst_daily_basin_summary_2020_2024.xlsx


,basin,time_utc,sst_mean,sst_min,sst_max,sst_std,anom_mean,err_mean,valid_grid_count
0,Arabian_Sea,2020-03-01T12:00:00Z,27.840880,23.439999,29.92,1.372615,0.570723,0.120060,3818
1,Arabian_Sea,2020-03-02T12:00:00Z,27.926555,23.650000,29.84,1.327094,0.630335,0.117978,3818
2,Arabian_Sea,2020-03-03T12:00:00Z,27.915413,23.600000,29.74,1.370349,0.593266,0.120351,3818
3,Arabian_Sea,2020-03-04T12:00:00Z,27.858452,23.609999,29.71,1.402710,0.510283,0.124835,3818
4,Arabian_Sea,2020-03-05T12:00:00Z,27.825450,23.619999,29.80,1.408405,0.451349,0.120893,3818


In [6]:
import shutil
from pathlib import Path

# Folder you want to zip
dataset_folder = Path("noaa_oisst_output")

# Output zip file name
zip_name = "noaa_oisst_output_zip"

# Create zip
shutil.make_archive(zip_name, "zip", dataset_folder)

print("ZIP file created:", zip_name + ".zip")

ZIP file created: noaa_oisst_output_zip.zip
